# Phase 4 - Notebook 03: MVSplat Architecture Deep Dive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase4/03_mvsplat_architecture.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand MVSplat's complete architecture end-to-end
2. Implement a simplified feature encoder (U-Net backbone)
3. Understand Cost Volume processing via 3D convolutions
4. Build Gaussian prediction heads and trace their tensor shapes
5. Walk through a complete forward pass with data flow visualization

**Estimated Time**: 90 minutes

**Prerequisites**: Notebooks 01 (Cost Volume), 02 (Pixel-aligned Gaussians)

---

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")

## 1. MVSplat Architecture Overview

MVSplat (ECCV 2024) achieves efficient feed-forward 3D Gaussian prediction using a **lightweight, explicit geometry** approach:

```
Input: 2 images + camera poses
                │
                ▼
┌─────────────────────────────────┐
│    Feature Encoder (U-Net)      │  Shared weights
│    Image → Multi-scale features │  across views
└──────────────┬──────────────────┘
               │ [B, C, H, W] per view
               ▼
┌─────────────────────────────────┐
│   Plane Sweeping Cost Volume    │  Explicit geometry
│   Warp features at D depths     │  reasoning
└──────────────┬──────────────────┘
               │ [B, C, D, H, W]
               ▼
┌─────────────────────────────────┐
│   Cost Volume Processor         │  3D Convolutions
│   (3D U-Net / 3D CNN)           │  or 3D U-Net
└──────────────┬──────────────────┘
               │ [B, C', H, W] processed features
               ▼
┌─────────────────────────────────┐
│   Gaussian Prediction Heads     │  Per-pixel prediction
│   Depth │ Scale │ Rot │ Opacity │
└──────────────┬──────────────────┘
               │
               ▼
┌─────────────────────────────────┐
│   Pixel-aligned Gaussians       │  Back-project + render
│   Positions from depth          │  from novel views
└─────────────────────────────────┘
```

**Key design choices:**
- **Explicit geometry** (Cost Volume) vs pixelSplat's implicit learning
- **Lightweight** 2D encoder + 3D cost volume processor
- **No Transformer** in the main pathway (faster than pixelSplat)
- **Two input views** (generalizes to more at inference)

In [ ]:
# Visualize the complete architecture

fig, ax = plt.subplots(1, 1, figsize=(16, 12))
ax.set_xlim(0, 16)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('MVSplat Architecture Overview', fontsize=16, fontweight='bold', pad=20)

# Helper function
def draw_box(ax, x, y, w, h, text, color='lightblue', fontsize=9, subtext=None):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='black', lw=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2 + (0.15 if subtext else 0), text,
            ha='center', va='center', fontsize=fontsize, fontweight='bold')
    if subtext:
        ax.text(x + w/2, y + h/2 - 0.2, subtext,
                ha='center', va='center', fontsize=7, style='italic', color='#444')

def draw_arrow(ax, x1, y1, x2, y2, text=None):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    if text:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx + 0.1, my, text, fontsize=7, color='#666', style='italic')

# Input images
draw_box(ax, 1, 10.5, 2.5, 0.8, 'Image 1', '#FFE0B2', subtext='[B,3,H,W]')
draw_box(ax, 5, 10.5, 2.5, 0.8, 'Image 2', '#FFE0B2', subtext='[B,3,H,W]')
draw_box(ax, 9.5, 10.5, 3.5, 0.8, 'Camera Poses', '#E0E0E0', subtext='K, R, t per view')

# Feature encoder (shared)
draw_box(ax, 1, 8.8, 6.5, 1.0, 'Feature Encoder (U-Net)', '#BBDEFB',
         subtext='Shared weights, multi-scale')
draw_arrow(ax, 2.25, 10.5, 2.25, 9.8)
draw_arrow(ax, 6.25, 10.5, 6.25, 9.8)

# Features
draw_box(ax, 1, 7.3, 2.5, 0.8, 'Feat Ref', '#C8E6C9', subtext='[B,C,H,W]')
draw_box(ax, 5, 7.3, 2.5, 0.8, 'Feat Src', '#C8E6C9', subtext='[B,C,H,W]')
draw_arrow(ax, 2.25, 8.8, 2.25, 8.1)
draw_arrow(ax, 6.25, 8.8, 6.25, 8.1)

# Cost Volume
draw_box(ax, 1, 5.7, 6.5, 1.0, 'Plane Sweeping Cost Volume', '#E1BEE7',
         subtext='Warp feat_src at D depth planes')
draw_arrow(ax, 2.25, 7.3, 3, 6.7)
draw_arrow(ax, 6.25, 7.3, 5.5, 6.7)
draw_arrow(ax, 11.25, 10.5, 11.25, 6.7, 'intrinsics +\nrelative pose')
ax.text(8.5, 6.2, '[B, C, D, H, W]', fontsize=8, color='#666', style='italic')

# Cost Volume Processor
draw_box(ax, 1, 4.1, 6.5, 1.0, 'Cost Volume Processor (3D CNN)', '#F8BBD0',
         subtext='Spatial + depth aggregation')
draw_arrow(ax, 4.25, 5.7, 4.25, 5.1)
ax.text(8.5, 4.6, '[B, C\', H, W]', fontsize=8, color='#666', style='italic')

# Prediction heads
draw_box(ax, 0.3, 2.5, 2.2, 0.9, 'Depth\nHead', '#B3E5FC', subtext='[B,1,H,W]')
draw_box(ax, 3.0, 2.5, 2.2, 0.9, 'Scale\nHead', '#B3E5FC', subtext='[B,3,H,W]')
draw_box(ax, 5.7, 2.5, 2.2, 0.9, 'Rotation\nHead', '#B3E5FC', subtext='[B,4,H,W]')
draw_box(ax, 8.4, 2.5, 2.2, 0.9, 'Opacity\nHead', '#B3E5FC', subtext='[B,1,H,W]')

draw_arrow(ax, 2, 4.1, 1.4, 3.4)
draw_arrow(ax, 3, 4.1, 4.1, 3.4)
draw_arrow(ax, 5, 4.1, 6.8, 3.4)
draw_arrow(ax, 6, 4.1, 9.5, 3.4)

# Output
draw_box(ax, 2.5, 0.8, 6, 1.0, 'Pixel-aligned Gaussians', '#A5D6A7',
         subtext='Back-project depth → 3D positions, merge params')
draw_arrow(ax, 1.4, 2.5, 4.5, 1.8)
draw_arrow(ax, 4.1, 2.5, 5.0, 1.8)
draw_arrow(ax, 6.8, 2.5, 5.5, 1.8)
draw_arrow(ax, 9.5, 2.5, 6.5, 1.8)

# Color from image
draw_box(ax, 11, 2.5, 2.2, 0.9, 'Image\nColor', '#FFF9C4', subtext='[B,3,H,W]')
draw_arrow(ax, 12.1, 2.5, 7.5, 1.8)

plt.tight_layout()
plt.show()

## 2. Feature Encoder

### 2.1 Design Choices

MVSplat uses a **lightweight U-Net encoder** that:
- Extracts multi-scale features from each input image
- **Shares weights** across all views (Siamese architecture)
- Outputs features at the same resolution as input (or 1/2, 1/4)
- Typical: 64 or 128 feature channels

The encoder is intentionally simple - the geometric reasoning happens in the Cost Volume, not in the features themselves.

### 2.2 Simplified Implementation

In [ ]:
class ConvBlock(nn.Module):
    """Basic convolution block: Conv2d + BN + ReLU."""
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride,
                      padding=kernel_size // 2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class SimpleUNetEncoder(nn.Module):
    """
    Simplified U-Net encoder for feature extraction.
    
    Architecture:
        Image [3,H,W] → Down1 [C,H/2,W/2] → Down2 [2C,H/4,W/4]
                                                    ↓
        Features [C,H,W] ← Up1 [C,H/2,W/2] ← Bottleneck [2C,H/4,W/4]
    """

    def __init__(self, in_channels=3, feature_channels=64):
        super().__init__()
        C = feature_channels

        # Encoder path (downsampling)
        self.enc1 = nn.Sequential(
            ConvBlock(in_channels, C),
            ConvBlock(C, C),
        )
        self.enc2 = nn.Sequential(
            ConvBlock(C, C * 2, stride=2),
            ConvBlock(C * 2, C * 2),
        )
        self.bottleneck = nn.Sequential(
            ConvBlock(C * 2, C * 2, stride=2),
            ConvBlock(C * 2, C * 2),
        )

        # Decoder path (upsampling)
        self.up2 = nn.ConvTranspose2d(C * 2, C * 2, 2, stride=2)
        self.dec2 = nn.Sequential(
            ConvBlock(C * 4, C * 2),  # concat with skip
            ConvBlock(C * 2, C),
        )
        self.up1 = nn.ConvTranspose2d(C, C, 2, stride=2)
        self.dec1 = nn.Sequential(
            ConvBlock(C * 2, C),  # concat with skip
            ConvBlock(C, C),
        )

    def forward(self, x):
        """Extract features. Input: [B, 3, H, W], Output: [B, C, H, W]."""
        # Encoder
        e1 = self.enc1(x)       # [B, C, H, W]
        e2 = self.enc2(e1)      # [B, 2C, H/2, W/2]
        bn = self.bottleneck(e2) # [B, 2C, H/4, W/4]

        # Decoder with skip connections
        d2 = self.up2(bn)       # [B, 2C, H/2, W/2]
        d2 = torch.cat([d2, e2], dim=1)  # [B, 4C, H/2, W/2]
        d2 = self.dec2(d2)      # [B, C, H/2, W/2]

        d1 = self.up1(d2)       # [B, C, H, W]
        d1 = torch.cat([d1, e1], dim=1)  # [B, 2C, H, W]
        d1 = self.dec1(d1)      # [B, C, H, W]

        return d1


# Test the encoder
encoder = SimpleUNetEncoder(in_channels=3, feature_channels=64)
dummy_img = torch.randn(2, 3, 64, 64)
features = encoder(dummy_img)

print(f"Input shape:   {list(dummy_img.shape)}")
print(f"Output shape:  {list(features.shape)}")
print(f"Parameters:    {sum(p.numel() for p in encoder.parameters()):,}")
print(f"\nThe encoder is SHARED across all input views (Siamese).")

In [ ]:
# Visualize feature extraction at each stage

encoder.eval()
torch.manual_seed(42)
test_img = torch.randn(1, 3, 64, 64)

# Hook to capture intermediate features
intermediates = {}
def hook_fn(name):
    def fn(module, input, output):
        intermediates[name] = output.detach()
    return fn

encoder.enc1.register_forward_hook(hook_fn('enc1'))
encoder.enc2.register_forward_hook(hook_fn('enc2'))
encoder.bottleneck.register_forward_hook(hook_fn('bottleneck'))
encoder.dec2.register_forward_hook(hook_fn('dec2'))
encoder.dec1.register_forward_hook(hook_fn('dec1'))

with torch.no_grad():
    output = encoder(test_img)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
stages = ['enc1', 'enc2', 'bottleneck', 'dec2', 'dec1']
titles = [
    f'Enc1\n{list(intermediates["enc1"].shape)}',
    f'Enc2\n{list(intermediates["enc2"].shape)}',
    f'Bottleneck\n{list(intermediates["bottleneck"].shape)}',
    f'Dec2\n{list(intermediates["dec2"].shape)}',
    f'Dec1 (output)\n{list(intermediates["dec1"].shape)}',
]

for ax, stage, title in zip(axes, stages, titles):
    feat = intermediates[stage][0]  # [C, H, W]
    # Show mean activation across channels
    feat_vis = feat.mean(dim=0).numpy()
    im = ax.imshow(feat_vis, cmap='viridis')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('U-Net Encoder: Feature Maps at Each Stage\n(mean activation across channels)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("The U-Net preserves spatial resolution while capturing multi-scale context.")
print("Skip connections transfer fine-grained details to the decoder.")

## 3. Cost Volume Construction

The Cost Volume is MVSplat's core geometric reasoning module. We covered the basics in Notebook 01; here we focus on its role in the full pipeline.

### 3.1 Recap: Cost Volume Dimensions

```
Cost Volume: [B, C, D, H, W]
                │  │  │  └── Spatial (image dimensions)
                │  │  └───── Depth (D hypotheses)
                │  └──────── Feature channels
                └─────────── Batch
```

For each pixel (h, w) at each depth hypothesis d:
- Warp the source feature map to the reference view
- Compute matching cost (feature difference squared)

In [ ]:
from src.feedforward.cost_volume import CostVolumeBuilder, depth_regression_softargmin

# Build a cost volume from encoder features
B, C, H, W = 1, 64, 16, 16
fx, fy = 30.0, 30.0

K = torch.tensor([[fx, 0, W/2], [0, fy, H/2], [0, 0, 1]], dtype=torch.float32)
K = K.unsqueeze(0)  # [1, 3, 3]

# Relative pose (small baseline)
angle = np.radians(3.0)
R = torch.tensor([[np.cos(angle), 0, np.sin(angle)],
                   [0, 1, 0],
                   [-np.sin(angle), 0, np.cos(angle)]], dtype=torch.float32)
T = torch.eye(4)
T[:3, :3] = R
T[:3, 3] = torch.tensor([0.3, 0.0, 0.0])
T = T.unsqueeze(0)  # [1, 4, 4]

# Simulated encoder features
torch.manual_seed(42)
feat_ref = torch.randn(B, C, H, W)
feat_src = torch.randn(B, C, H, W)

# Build cost volume
cv_builder = CostVolumeBuilder(
    num_depths=32,
    min_depth=2.0,
    max_depth=10.0,
    sampling='log_uniform'
)

cost_volume = cv_builder(feat_ref, [feat_src], K, [K], [T])

print(f"Feature shapes: ref={list(feat_ref.shape)}, src={list(feat_src.shape)}")
print(f"Cost volume shape: {list(cost_volume.shape)}  (B, C, D, H, W)")
print(f"Cost volume memory: {cost_volume.numel() * 4 / 1024:.1f} KB")
print(f"\nDepth planes: {cv_builder.depth_planes.shape[0]}")
print(f"Depth range: [{cv_builder.depth_planes[0]:.2f}, {cv_builder.depth_planes[-1]:.2f}]")

## 4. Cost Volume Processing (3D CNN)

### 4.1 Why Process the Cost Volume?

The raw cost volume is noisy because:
- Feature matching is imperfect
- Textureless regions produce ambiguous costs
- Occlusions create false matches

A 3D CNN **regularizes** the cost volume by:
- Smoothing cost spatially (neighboring pixels should have similar depth)
- Aggregating information along the depth dimension
- Learning to resolve ambiguities

### 4.2 Architecture Options

| Method | Architecture | Parameters | Speed |
|--------|-------------|------------|-------|
| MVSNet | 3D U-Net | ~1M | Moderate |
| CasMVSNet | Cascade 3D CNN | ~0.5M | Faster |
| MVSplat | Lightweight 3D CNN | ~0.3M | Fast |

### 4.3 Simplified 3D CNN Implementation

In [ ]:
class CostVolumeProcessor(nn.Module):
    """
    3D CNN to regularize the cost volume.
    
    Takes raw cost volume [B, C, D, H, W] and produces
    processed features [B, C', H, W] for Gaussian prediction.
    
    Architecture:
        3D Conv (reduce D) → 3D Conv → Collapse depth → 2D Conv
    """

    def __init__(self, in_channels=64, hidden_channels=32, out_channels=64):
        super().__init__()

        # 3D convolutions for spatial-depth processing
        self.conv3d_1 = nn.Sequential(
            nn.Conv3d(in_channels, hidden_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(hidden_channels),
            nn.ReLU(inplace=True),
        )
        self.conv3d_2 = nn.Sequential(
            nn.Conv3d(hidden_channels, hidden_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(hidden_channels),
            nn.ReLU(inplace=True),
        )
        # Collapse depth dimension: 3D → 2D
        self.depth_collapse = nn.Conv3d(
            hidden_channels, hidden_channels,
            kernel_size=(3, 1, 1), padding=(1, 0, 0)
        )

        # Final 2D refinement
        self.conv2d = nn.Sequential(
            nn.Conv2d(hidden_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, cost_volume):
        """
        Process cost volume.
        
        Input:  [B, C, D, H, W] raw cost volume
        Output: [B, C', H, W] processed features + [B, 1, D, H, W] depth prob
        """
        B, C, D, H, W = cost_volume.shape

        # 3D convolutions
        x = self.conv3d_1(cost_volume)   # [B, 32, D, H, W]
        x = self.conv3d_2(x)             # [B, 32, D, H, W]

        # Collapse depth via pooling
        x_collapsed = self.depth_collapse(x)  # [B, 32, D, H, W]
        x_2d = x_collapsed.mean(dim=2)        # [B, 32, H, W] - average over depth

        # Refine with 2D conv
        features = self.conv2d(x_2d)     # [B, C', H, W]

        return features


# Test the processor
processor = CostVolumeProcessor(in_channels=64, hidden_channels=32, out_channels=64)

processed_features = processor(cost_volume)

print(f"Cost volume input:  {list(cost_volume.shape)}")
print(f"Processed features: {list(processed_features.shape)}")
print(f"\nProcessor parameters: {sum(p.numel() for p in processor.parameters()):,}")
print(f"\nThe 3D CNN regularizes the raw cost and produces 2D features")
print(f"that encode geometric information (depth-aware features).")

In [ ]:
# Visualize: raw cost volume vs processed features

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

# Top row: raw cost volume slices (averaged over feature channels)
raw_cost = cost_volume[0].mean(dim=0).detach()  # [D, H, W]
D = raw_cost.shape[0]
slice_ids = [0, D//4, D//2, 3*D//4]

for i, si in enumerate(slice_ids):
    ax = axes[0, i]
    d_val = cv_builder.depth_planes[si].item()
    im = ax.imshow(raw_cost[si].numpy(), cmap='RdYlGn_r')
    ax.set_title(f'Raw Cost @ d={d_val:.1f}', fontsize=9, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

# Bottom row: processed features (first 4 channels)
proc_feat = processed_features[0].detach()  # [C', H, W]
for i in range(4):
    ax = axes[1, i]
    im = ax.imshow(proc_feat[i].numpy(), cmap='viridis')
    ax.set_title(f'Processed Feature ch={i}', fontsize=9, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

axes[0, 0].set_ylabel('Raw Cost Volume\n(noisy)', fontsize=10)
axes[1, 0].set_ylabel('Processed Features\n(regularized)', fontsize=10)

plt.suptitle('Cost Volume Processing: From Noisy Costs to Smooth Features',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Gaussian Prediction Heads

### 5.1 From Features to Gaussian Parameters

The processed features are fed to **separate prediction heads**, each producing one Gaussian property:

```
Processed Features [B, C', H, W]
          │
    ┌─────┼─────┬────────┬────────┐
    ▼     ▼     ▼        ▼        ▼
  Depth  Scale  Rot    Opacity  (Color)
 [B,1]  [B,3]  [B,4]  [B,1]   from img
    │     │     │        │        │
    ▼     ▼     ▼        ▼        ▼
  (none) exp  normalize sigmoid  (none)
    │     │     │        │        │
    └─────┴─────┴────────┴────────┘
                │
                ▼
    Pixel-aligned Gaussians
```

In [ ]:
from src.feedforward.gaussian_predictor import (
    GaussianPredictionHeads,
    DepthHead,
    CovarianceHead,
    OpacityHead,
)

# Create prediction heads
heads = GaussianPredictionHeads(
    in_channels=64,
    hidden_channels=32,
    depth_mode='regression',
    covariance_mode='3d',
)

# Predict Gaussian parameters from processed features
with torch.no_grad():
    predictions = heads(processed_features)

print("Gaussian Prediction Heads output:")
print(f"{'Parameter':12s} | {'Shape':20s} | {'Range':30s} | Activation")
print("-" * 85)

activations = {
    'depth': 'Softplus (positive)',
    'scales': 'exp + min_scale (positive)',
    'rotations': 'L2 normalize (unit quaternion)',
    'opacities': 'sigmoid ([0, 1])',
}

for key, val in predictions.items():
    rng = f'[{val.min():.4f}, {val.max():.4f}]'
    act = activations.get(key, 'none')
    print(f"{key:12s} | {str(list(val.shape)):20s} | {rng:30s} | {act}")

print(f"\nTotal head parameters: {sum(p.numel() for p in heads.parameters()):,}")

In [ ]:
# Visualize predicted Gaussian parameters

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Depth
ax = axes[0, 0]
depth_pred = predictions['depth'][0, 0].detach().numpy()
im = ax.imshow(depth_pred, cmap='plasma')
ax.set_title('Predicted Depth', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.axis('off')

# Scales (3 channels)
for i in range(3):
    ax = axes[0, 1] if i == 0 else axes[0, 2] if i == 1 else axes[1, 0]
    scale = predictions['scales'][0, i].detach().numpy()
    im = ax.imshow(scale, cmap='magma')
    ax.set_title(f'Scale (axis {["x","y","z"][i]})', fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.axis('off')

# Opacity
ax = axes[1, 1]
opacity = predictions['opacities'][0, 0].detach().numpy()
im = ax.imshow(opacity, cmap='gray', vmin=0, vmax=1)
ax.set_title('Predicted Opacity', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.axis('off')

# Rotation magnitude
ax = axes[1, 2]
rot = predictions['rotations'][0].detach()  # [4, H, W]
rot_w = rot[0].numpy()  # quaternion w component
im = ax.imshow(rot_w, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_title('Rotation (w component)', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.axis('off')

plt.suptitle('Predicted Gaussian Parameters (Per-pixel)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Each pixel has its own Gaussian parameters.")
print(f"Total Gaussians per view: {H} x {W} = {H * W}")

## 6. Complete Forward Pass

### 6.1 Putting It All Together

Let's build a simplified but complete MVSplat model and trace a forward pass.

In [ ]:
from src.feedforward.pixel_aligned import PixelAlignedGaussians, unproject_depth_to_3d


class SimplifiedMVSplat(nn.Module):
    """
    Simplified MVSplat architecture for educational purposes.
    
    Full pipeline: images → features → cost volume → process → predict → Gaussians
    """

    def __init__(
        self,
        feature_channels=64,
        num_depths=32,
        min_depth=2.0,
        max_depth=10.0,
    ):
        super().__init__()

        # 1. Feature encoder (shared across views)
        self.encoder = SimpleUNetEncoder(
            in_channels=3,
            feature_channels=feature_channels,
        )

        # 2. Cost volume builder
        self.cost_volume_builder = CostVolumeBuilder(
            num_depths=num_depths,
            min_depth=min_depth,
            max_depth=max_depth,
            sampling='log_uniform',
        )

        # 3. Cost volume processor (3D CNN)
        self.cv_processor = CostVolumeProcessor(
            in_channels=feature_channels,
            hidden_channels=32,
            out_channels=feature_channels,
        )

        # 4. Gaussian prediction heads
        self.gaussian_heads = GaussianPredictionHeads(
            in_channels=feature_channels,
            hidden_channels=32,
            depth_mode='regression',
            covariance_mode='3d',
        )

    def forward(self, images, K, poses, T_relative):
        """
        Full MVSplat forward pass.
        
        Args:
            images: list of [B, 3, H, W] input images (2 views)
            K: [B, 3, 3] camera intrinsics (shared)
            poses: list of [B, 4, 4] camera-to-world poses
            T_relative: [B, 4, 4] ref-to-source transform
        
        Returns:
            PixelAlignedGaussians for each input view
        """
        shapes = {}  # Track tensor shapes

        # Step 1: Extract features (shared encoder)
        features = []
        for i, img in enumerate(images):
            feat = self.encoder(img)
            features.append(feat)
            shapes[f'feat_view{i}'] = list(feat.shape)

        # Step 2: Build cost volume (using view 0 as reference)
        cost_vol = self.cost_volume_builder(
            features[0], [features[1]], K, [K], [T_relative]
        )
        shapes['cost_volume'] = list(cost_vol.shape)

        # Step 3: Process cost volume
        proc_features = self.cv_processor(cost_vol)
        shapes['processed_features'] = list(proc_features.shape)

        # Step 4: Predict Gaussian parameters
        predictions = self.gaussian_heads(proc_features)
        for key, val in predictions.items():
            shapes[f'pred_{key}'] = list(val.shape)

        # Step 5: Create pixel-aligned Gaussians
        gaussians = PixelAlignedGaussians.from_depth_and_features(
            depth=predictions['depth'],
            features={
                'scales': predictions['scales'],
                'rotations': predictions['rotations'],
                'opacities': predictions['opacities'],
            },
            K=K,
            pose=poses[0],
            image_colors=images[0],
        )
        shapes['gaussians_positions'] = list(gaussians.positions.shape)

        return gaussians, predictions, shapes


# Create the model
model = SimplifiedMVSplat(
    feature_channels=64,
    num_depths=32,
    min_depth=2.0,
    max_depth=10.0,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"SimplifiedMVSplat created")
print(f"Total parameters: {total_params:,}")
print(f"\nComponent breakdown:")
for name, module in [
    ('Encoder', model.encoder),
    ('CV Processor', model.cv_processor),
    ('Gaussian Heads', model.gaussian_heads),
]:
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:20s}: {n:>8,} params ({100*n/total_params:.1f}%)")

In [ ]:
# Run a complete forward pass and trace shapes

model.eval()
torch.manual_seed(42)

B, H, W = 1, 32, 32
images = [torch.randn(B, 3, H, W), torch.randn(B, 3, H, W)]
K_batch = torch.tensor([[30, 0, W/2], [0, 30, H/2], [0, 0, 1]],
                        dtype=torch.float32).unsqueeze(0)
pose0 = torch.eye(4).unsqueeze(0)
pose1 = T.clone()

with torch.no_grad():
    gaussians, predictions, shapes = model(
        images, K_batch, [pose0, pose1], T
    )

print("=" * 60)
print("COMPLETE FORWARD PASS - Tensor Shape Trace")
print("=" * 60)
for stage, shape in shapes.items():
    print(f"  {stage:25s}: {shape}")

print(f"\n{'=' * 60}")
print(f"OUTPUT: {gaussians}")
print(f"  Positions range: [{gaussians.positions.min():.2f}, {gaussians.positions.max():.2f}]")
print(f"  Scales range:    [{gaussians.scales.min():.4f}, {gaussians.scales.max():.4f}]")
print(f"  Opacities range: [{gaussians.opacities.min():.4f}, {gaussians.opacities.max():.4f}]")

In [ ]:
# Visualize the forward pass data flow

fig, axes = plt.subplots(2, 5, figsize=(22, 8))

# Row 1: Pipeline stages
# Input image
ax = axes[0, 0]
img_vis = images[0][0].permute(1, 2, 0).numpy()
img_vis = (img_vis - img_vis.min()) / (img_vis.max() - img_vis.min())
ax.imshow(img_vis)
ax.set_title('1. Input Image\n[B,3,H,W]', fontsize=9, fontweight='bold')
ax.axis('off')

# Encoder features
ax = axes[0, 1]
with torch.no_grad():
    enc_feat = model.encoder(images[0])
ax.imshow(enc_feat[0].mean(dim=0).numpy(), cmap='viridis')
ax.set_title('2. Encoder Features\n[B,64,H,W]', fontsize=9, fontweight='bold')
ax.axis('off')

# Cost volume (one slice)
ax = axes[0, 2]
with torch.no_grad():
    cv = model.cost_volume_builder(enc_feat, [model.encoder(images[1])], K_batch, [K_batch], [T])
cv_mid = cv[0].mean(dim=0)[cv.shape[2]//2].numpy()
ax.imshow(cv_mid, cmap='RdYlGn_r')
ax.set_title('3. Cost Volume (mid slice)\n[B,64,D,H,W]', fontsize=9, fontweight='bold')
ax.axis('off')

# Processed features
ax = axes[0, 3]
with torch.no_grad():
    proc = model.cv_processor(cv)
ax.imshow(proc[0].mean(dim=0).numpy(), cmap='viridis')
ax.set_title('4. Processed Features\n[B,64,H,W]', fontsize=9, fontweight='bold')
ax.axis('off')

# Predicted depth
ax = axes[0, 4]
ax.imshow(predictions['depth'][0, 0].numpy(), cmap='plasma')
ax.set_title('5. Predicted Depth\n[B,1,H,W]', fontsize=9, fontweight='bold')
ax.axis('off')

# Row 2: Predicted Gaussian properties
prop_data = [
    ('Scale (x)', predictions['scales'][0, 0].numpy(), 'magma'),
    ('Scale (y)', predictions['scales'][0, 1].numpy(), 'magma'),
    ('Scale (z)', predictions['scales'][0, 2].numpy(), 'magma'),
    ('Opacity', predictions['opacities'][0, 0].numpy(), 'gray'),
    ('Rotation (w)', predictions['rotations'][0, 0].numpy(), 'coolwarm'),
]

for i, (title, data, cmap) in enumerate(prop_data):
    ax = axes[1, i]
    ax.imshow(data, cmap=cmap)
    ax.set_title(f'{title}', fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle('MVSplat Forward Pass: Data Flow Visualization',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Multi-View Processing

In practice, MVSplat processes each input view as a reference and produces Gaussians from each. The final Gaussian set is the union:

```
View 1 as ref → H×W Gaussians ──┐
                                  ├──► Concatenate ──► Total: 2×H×W Gaussians
View 2 as ref → H×W Gaussians ──┘                         ──► Render
```

Each view "sees" the scene from its perspective, producing complementary Gaussians.

In [ ]:
# Multi-view Gaussian merging

model.eval()

with torch.no_grad():
    # Process view 0 as reference
    gaussians_v0, _, _ = model(images, K_batch, [pose0, pose1], T)
    
    # Process view 1 as reference (swap images and invert transform)
    T_inv = torch.inverse(T)
    gaussians_v1, _, _ = model(
        [images[1], images[0]], K_batch, [pose1, pose0], T_inv
    )

# Merge
gaussians_merged = gaussians_v0.merge(gaussians_v1)

print(f"View 0 Gaussians: {gaussians_v0.num_gaussians}")
print(f"View 1 Gaussians: {gaussians_v1.num_gaussians}")
print(f"Merged Gaussians: {gaussians_merged.num_gaussians}")

# Visualize merged point cloud
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (pag, title, color) in enumerate([
    (gaussians_v0, 'View 0 Gaussians', 'blue'),
    (gaussians_v1, 'View 1 Gaussians', 'red'),
    (gaussians_merged, 'Merged', None),
]):
    ax = axes[idx]
    pos = pag.positions[0].numpy()
    
    if color:
        ax.scatter(pos[:, 0], pos[:, 2], c=color, s=5, alpha=0.5)
    else:
        n0 = gaussians_v0.num_gaussians
        ax.scatter(pos[:n0, 0], pos[:n0, 2], c='blue', s=3, alpha=0.4, label='View 0')
        ax.scatter(pos[n0:, 0], pos[n0:, 2], c='red', s=3, alpha=0.4, label='View 1')
        ax.legend(fontsize=8)
    
    ax.set_title(f'{title} (N={pag.num_gaussians})', fontsize=11, fontweight='bold')
    ax.set_xlabel('X'); ax.set_ylabel('Z (depth)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Comparison with Official MVSplat

Our simplified version captures the key ideas. Here's how it compares to the real implementation:

| Component | Our Simplified | Official MVSplat |
|-----------|---------------|------------------|
| Encoder | Simple U-Net | Pre-trained backbone (e.g., ResNet + FPN) |
| Feature dim | 64 | 128-256 |
| Cost Volume | Basic plane sweep | Cascaded with coarse-to-fine |
| 3D Processing | Simple 3D CNN | 3D U-Net with skip connections |
| Depth | Direct regression | Soft argmin from cost volume |
| Multi-view | Sequential | Parallel + cross-view attention |
| Rendering | Not included | gsplat / diff-gaussian-rasterization |
| Training | Not included | Photometric loss + LPIPS |
| Parameters | ~300K | ~10M |

### Key Takeaways

1. **Architecture is modular**: encoder → cost volume → processor → heads
2. **Cost Volume is the key differentiator** from pixelSplat
3. **Lightweight design**: No Transformer, fast inference
4. **Explicit geometry**: Cost Volume encodes multi-view geometry directly

In [ ]:
# Summary

summary = """
=====================================================================
     Notebook 03 Summary: MVSplat Architecture Deep Dive
=====================================================================

1. FEATURE ENCODER (U-Net)
   - Shared weights across views (Siamese)
   - Multi-scale features with skip connections
   - Output: [B, C, H, W] per view

2. COST VOLUME (Plane Sweeping)
   - Warp source features at D depth hypotheses
   - Compute matching cost: ||F_ref - F_src_warped||^2
   - Output: [B, C, D, H, W]

3. COST VOLUME PROCESSOR (3D CNN)
   - Regularize noisy cost volume
   - 3D convolutions for spatial + depth aggregation
   - Collapse depth dimension to 2D features
   - Output: [B, C', H, W]

4. GAUSSIAN PREDICTION HEADS
   - Depth: Softplus activation (positive)
   - Scale: exp activation (positive)
   - Rotation: L2 normalize (unit quaternion)
   - Opacity: sigmoid ([0, 1])
   - Color: from input image directly

5. PIXEL-ALIGNED GAUSSIANS
   - Back-project predicted depth to 3D positions
   - One Gaussian per pixel per view
   - Merge views by concatenation

=====================================================================
"""
print(summary)

## What's Next?

**[04_pixelsplat_implicit_geometry.ipynb](./04_pixelsplat_implicit_geometry.ipynb)** - pixelSplat: an alternative approach without explicit Cost Volume, using cross-attention for implicit geometry learning.

---

## References

1. MVSplat: https://arxiv.org/abs/2403.14627
2. MVSNet: https://arxiv.org/abs/1804.02505
3. CasMVSNet: https://arxiv.org/abs/1912.06378